In [69]:
# Create data file with 20k samples using uniform distributions
# over T, P, qc, and qv multiplier
%run ../SatAdj_GenerateSamples.py -f samples20k.csv -N 20000 -v

Summary stats
	mean delta T: 0.1734959264791221
	variance in delta T: 1.2935450525645755
	max delta T: 12.37025931556093
	min delta T: -36.16707639600628
	min |delta T|: 2.87798229692271e-10 

	mean delta qv: -6.971066325931129e-05
	variance in delta qv: 2.0883410437199539e-07
	max delta qv: 2.0883410437199539e-07
	min delta qv: -0.004970370192992388
	min |delta qv|: 1.1563232750919434e-13 

	mean delta qc: 6.971066325931129e-05
	variance in delta qc: 2.0883410437199539e-07
	max delta qc: 0.004970370192992387
	min delta qc: -0.01453193129591533
	min |delta qc|: 1.1563233009992402e-13


In [70]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler, QuantileTransformer

In [71]:
df = pd.read_csv('samples20k.csv')

In [72]:
X = df[['T_in', 'qv_in', 'qc_in', 'pres_in']].values
Y = df[['T_out', 'qv_out', 'qc_out']].values

In [73]:
# Use quantile transformer because we drew from uniform distributions for T, P, qc.
# Dependence of qv on T means this isn't strictly reasonable.
scaler_x = QuantileTransformer()
scaler_y = QuantileTransformer()

In [74]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.30)

X_train_tensor = torch.tensor(X_train, dtype=torch.float64)
X_test_tensor = torch.tensor(X_test, dtype=torch.float64)
Y_train_tensor = torch.tensor(Y_train, dtype=torch.float64)
Y_test_tensor = torch.tensor(Y_test, dtype=torch.float64)

In [75]:
X_train_scaled = torch.tensor(scaler_x.fit_transform(X_train_tensor), dtype=torch.float64)
X_test_scaled = torch.tensor(scaler_x.fit_transform(X_test_tensor), dtype=torch.float64)
Y_train_scaled = torch.tensor(scaler_y.fit_transform(Y_train_tensor), dtype=torch.float64)
Y_test_scaled = torch.tensor(scaler_y.fit_transform(Y_test_tensor), dtype=torch.float64)

In [76]:
X_train_scaled

tensor([[0.5044, 0.7329, 0.9998, 0.0543],
        [0.7902, 0.8021, 0.9697, 0.4414],
        [0.3783, 0.5228, 0.3917, 0.2080],
        ...,
        [0.6327, 0.7801, 0.8137, 0.1529],
        [0.1714, 0.1528, 0.9796, 0.9231],
        [0.0197, 0.0345, 0.5083, 0.9344]], dtype=torch.float64)

In [77]:
class NeuralNet(nn.Module):
    def __init__(self, n_in, n_out, width):
        super(NeuralNet, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(n_in, width),
            nn.ReLU(),
            nn.Linear(width, width),
            nn.ReLU(),
            nn.Linear(width, n_out)
        )

    def forward(self, x):
        return self.model(x)

In [78]:
n_in = 4
n_out = 3
width = 64
m_tol = 1e-6

net = NeuralNet(n_in, n_out, width)
net.double()
criterion = nn.MSELoss()
optimizer = optim.Adam(net.parameters())

In [ ]:
epochs = 5000
for epoch in range(epochs):
    net.train()
    
    pred = net(X_train_scaled)
    loss = criterion(pred, Y_train_scaled)

    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.3e}")

    if (loss.item() < m_tol) :
        print(f"Breaking at Epoch {epoch}, Loss: {loss.item():.3e}")
        break

Epoch 0, Loss: 4.340e-01
Epoch 100, Loss: 3.922e-03
Epoch 200, Loss: 2.271e-03
Epoch 300, Loss: 1.823e-03
Epoch 400, Loss: 1.673e-03
Epoch 500, Loss: 1.582e-03
Epoch 600, Loss: 1.518e-03
Epoch 700, Loss: 1.468e-03
Epoch 800, Loss: 1.425e-03
Epoch 900, Loss: 1.385e-03
Epoch 1000, Loss: 1.347e-03
Epoch 1100, Loss: 1.310e-03
Epoch 1200, Loss: 1.272e-03
Epoch 1300, Loss: 1.230e-03
Epoch 1400, Loss: 1.179e-03
Epoch 1500, Loss: 1.125e-03
Epoch 1600, Loss: 1.074e-03
Epoch 1700, Loss: 1.022e-03
Epoch 1800, Loss: 9.809e-04
Epoch 1900, Loss: 9.386e-04
Epoch 2000, Loss: 9.039e-04
Epoch 2100, Loss: 8.768e-04
Epoch 2200, Loss: 8.459e-04
Epoch 2300, Loss: 8.216e-04
Epoch 2400, Loss: 7.991e-04
Epoch 2500, Loss: 7.812e-04
Epoch 2600, Loss: 7.611e-04
Epoch 2700, Loss: 7.462e-04
Epoch 2800, Loss: 7.279e-04
Epoch 2900, Loss: 7.235e-04
Epoch 3000, Loss: 6.991e-04
Epoch 3100, Loss: 6.857e-04
Epoch 3200, Loss: 6.748e-04
Epoch 3300, Loss: 6.602e-04
Epoch 3400, Loss: 6.497e-04
Epoch 3500, Loss: 6.379e-04
Epoc

In [ ]:
net.eval()
with torch.no_grad():
    Y_pred = net(X_test_scaled)
    mse_loss = nn.MSELoss()

    loss_total = mse_loss(Y_pred, Y_test_scaled)
    print(f"Total test loss (scaled): {loss_total.item():.3e}")
    
    loss_T = mse_loss(Y_pred[:,0], Y_test_scaled[:,0])
    print(f"Scaled Test MSE T: {loss_T.item():.3e}")

    loss_qv = mse_loss(Y_pred[:,1], Y_test_scaled[:,1])
    print(f"Scaled Test MSE qv: {loss_qv.item():.3e}")

    loss_qc = mse_loss(Y_pred[:,2], Y_test_scaled[:,2])
    print(f"Scaled Test MSE qc: {loss_qc.item():.3e}")

In [ ]:
Y_pred_unscaled = torch.tensor(scaler_y.inverse_transform(Y_pred), dtype=torch.float64)

loss_total_unscaled = mse_loss(Y_pred_unscaled, Y_test_tensor)
print(f"Total test loss (unscaled): {loss_total_unscaled:.3e}")

loss_T_unscaled = mse_loss(Y_pred_unscaled[:,0], Y_test_tensor[:,0])
print(f"Unscaled test MSE T: {loss_T_unscaled:.3e}")

loss_qv_unscaled = mse_loss(Y_pred_unscaled[:,1], Y_test_tensor[:,1])
print(f"Unscaled test MSE qv: {loss_qv_unscaled:.3e}")

loss_qc_unscaled = mse_loss(Y_pred_unscaled[:,2], Y_test_tensor[:,2])
print(f"Unscaled test MSE qc: {loss_qc_unscaled:.3e}")